# Training: Null Models (Reviewer 2, Comment 2)

**Scope:**

- **Subjects:** VALID_SUBJECTS, [1, 2, 4, 6, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22]
- **Channel:** Fz
- **Time window:** 2 s
- **CV:** GroupKFold, 5 folds over 40 trials
- **Train overlap:** 50% | **Test overlap:** 0%

**Purpose:**

Establish reference points for the predictions reported in Section 4.5.
Two null models are computed under the **same CV folds** as the real
between-participant model:

1. **Shuffled pairing** — the network is trained and tested with video
   paired to the EEG of a *different* trial of the same subject. This
   destroys the true trial-level video-EEG correspondence while keeping
   every other aspect of the pipeline identical (network, loss,
   optimizer, folds, overlap, seeds).

2. **Trivial predictors** — no training:
   - `const_zero`       : predicts 0 (equal to the mean of z-scored
                          targets; RMSE = 1 by construction)
   - `const_train_mean` : predicts the training-set scalar mean
   - `mean_waveform`    : predicts the training-set mean waveform

**Settings (match the real between-participant model):**

- 10 epochs, batch size 16
- Adam: lr = 1e-4, weight_decay = 1e-5
- LR decayed by 0.1 every 5 epochs
- Train overlap 50%, test overlap 0%

**Runs:**

- Shuffled pairing, per subject — exp start 500 + 10·index
- Trivial baselines, per subject — no training, no checkpoints

**Output directories:**

- `Null-Models/Shuffled/Results/`
- `Null-Models/Trivial/Results/`
- `Null-Models/Shuffled/Models/`

In [ ]:
# ============================================================
# 0. Environment setup (Colab or local)
# ============================================================
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
    !pip install -q pims


# ============================================================
# 1. Standard library
# ============================================================
import os
import sys
import pickle
import traceback

# ============================================================
# 2. Third-party
# ============================================================
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# 3. Project paths
# ============================================================
if IN_COLAB:
    SOURCE_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Source'
    DATA_ROOT   = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Data'
else:
    SOURCE_ROOT = r'D:\Thesis\Codes\Source'
    DATA_ROOT   = r'D:\Thesis\Codes\Data'

if not os.path.isdir(os.path.join(SOURCE_ROOT, 'face2eeg')):
    raise FileNotFoundError(
        f"face2eeg package not found under SOURCE_ROOT = {SOURCE_ROOT}. "
        f"Expected directory: {os.path.join(SOURCE_ROOT, 'face2eeg')}"
    )

if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)


# ============================================================
# 4. Package imports
# ============================================================
from face2eeg.core.experiments import run_kfold_pipeline, run_trivial_baselines
from face2eeg.core.models      import try_gpu
from face2eeg.core.data        import clear_video_cache, clear_signals_cache

# Framework constants (from config)
from face2eeg.core.config import (
    SEED,
    FS_EEG,
    FS_VIDEO,
    VALID_SUBJECTS
)


# ============================================================
# 5. Reproducibility
# ============================================================
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================================
# Configuration
# ============================================================
BASE = os.path.join(DATA_ROOT, 'Experiments', 'Cross-Validation')

EXP_DIR = os.path.join(BASE, 'Null-Models')


# ---- Shuffled-pairing null model ----
SHUFFLED_DIR         = os.path.join(EXP_DIR, 'Shuffled')
RESULTS_DIR_SHUFFLED = os.path.join(SHUFFLED_DIR, 'Results')
MODELS_DIR_SHUFFLED  = os.path.join(SHUFFLED_DIR, 'Models')


# ---- Trivial baselines (no training, no checkpoints) ----
TRIVIAL_DIR         = os.path.join(EXP_DIR, 'Trivial')
RESULTS_DIR_TRIVIAL = os.path.join(TRIVIAL_DIR, 'Results')


# ---- Create all directories ----
for _d in (EXP_DIR,
           SHUFFLED_DIR, RESULTS_DIR_SHUFFLED, MODELS_DIR_SHUFFLED,
           TRIVIAL_DIR,  RESULTS_DIR_TRIVIAL):
    os.makedirs(_d, exist_ok=True)


# ============================================================
# Experiment scope
# ============================================================
SUBJECTS = VALID_SUBJECTS
CHANNEL  = 'Fz'


# ============================================================
# Hyperparameters (match the real between-participant model)
# ============================================================
TIME_WINDOW  = 2
FRAME_SIZE   = (32, 32)

EPOCHS       = 10
BATCH_SIZE   = 16
LR           = 1e-4
WEIGHT_DECAY = 1e-5
SCHED_STEP   = 5
SCHED_GAMMA  = 0.1

# Data generation (50% overlap train, 0% test)
TRAIN_OVERLAP       = True
TRAIN_OVERLAP_RATIO = 0.5
TEST_OVERLAP        = False
TEST_OVERLAP_RATIO  = 0.0

# Shuffled-pairing null model
SHUFFLE_SEED = SEED

# Unique exp_start per subject for the shuffled model
EXP_START_BASE = 500
EXP_START_PER_SUBJECT = {
    sid: EXP_START_BASE + i * 10 for i, sid in enumerate(SUBJECTS)
}


# ============================================================
# Device
# ============================================================
device = try_gpu()


# ============================================================
# Sanity check
# ============================================================
print("EXP_DIR            :", EXP_DIR)
print()
print("SHUFFLED_DIR       :", SHUFFLED_DIR)
print("RESULTS_DIR_SHUFFLED:", RESULTS_DIR_SHUFFLED)
print("MODELS_DIR_SHUFFLED :", MODELS_DIR_SHUFFLED)
print()
print("TRIVIAL_DIR        :", TRIVIAL_DIR)
print("RESULTS_DIR_TRIVIAL:", RESULTS_DIR_TRIVIAL)
print()
print("SUBJECTS    :", SUBJECTS)
print("CHANNEL     :", CHANNEL)
print("TIME_WINDOW :", TIME_WINDOW)
print("FRAME_SIZE  :", FRAME_SIZE)
print()
print("EPOCHS      :", EPOCHS)
print("BATCH_SIZE  :", BATCH_SIZE)
print("LR          :", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("SCHED_STEP  :", SCHED_STEP)
print("SCHED_GAMMA :", SCHED_GAMMA)
print()
print("TRAIN_OVERLAP       :", TRAIN_OVERLAP)
print("TRAIN_OVERLAP_RATIO :", TRAIN_OVERLAP_RATIO)
print("TEST_OVERLAP        :", TEST_OVERLAP)
print()
print("SHUFFLE_SEED        :", SHUFFLE_SEED)
print("EXP_START_PER_SUBJECT:")
for sid in SUBJECTS:
    print(f"  S{sid:02d}  ->  exp_start = {EXP_START_PER_SUBJECT[sid]}")
print()
print("FS_EEG      :", FS_EEG, " (from config)")
print("FS_VIDEO    :", FS_VIDEO, " (from config)")
print("device      :", device)

## Part 1 — Trivial baselines (no training)

Computes three trivial predictors on the test set under the same
GroupKFold splits as the real model. No training, no video decoding
(signal-only loading).

In [ ]:
for subject_id in SUBJECTS:
    print(f"\n{'#'*70}")
    print(f"#  TRIVIAL — SUBJECT {subject_id:02d}/{SUBJECTS[-1]:02d}")
    print(f"{'#'*70}")

    clear_signals_cache()

    result_path = os.path.join(
        RESULTS_DIR_TRIVIAL, f'sub{subject_id:02d}_ch{CHANNEL}.pkl'
    )
    if os.path.isfile(result_path):
        print(f"  [skip] already exists: {result_path}")
        continue

    try:
        res_trivial = run_trivial_baselines(
            subject_id=subject_id,
            channel_name=CHANNEL,
            time_window=TIME_WINDOW,

            # CV splits
            n_points=40, n_splits=5,

            # scaler
            scaler_method='z-score',

            # data generation
            signal_fs=FS_EEG,
            y_type='RawEEG',
            signal_normalize=True,
            # train
            train_overlap=TRAIN_OVERLAP,                
            train_overlap_ratio=TRAIN_OVERLAP_RATIO,    
            # test
            test_overlap=TEST_OVERLAP,
            test_overlap_ratio=TEST_OVERLAP_RATIO,

            verbose=True,
            root_dir=DATA_ROOT,
        )

        with open(result_path, 'wb') as f:
            pickle.dump(res_trivial, f)
        print(f"  [saved] {result_path}")

    except Exception as e:
        print(f"  !! ERROR for subject={subject_id}: {e}")
        traceback.print_exc()
        continue

## Part 2 — Shuffled-pairing null model

Trains the full pipeline with video and EEG paired across *different*
trials of the same subject. Everything else — network, loss, optimizer,
epochs, folds, overlap, seeds — is identical to the real model.

In [ ]:
results_shuffled = {}   # {subject_id: results_dict}

for subject_id in SUBJECTS:
    print(f"\n{'#'*70}")
    print(f"#  SHUFFLED — SUBJECT {subject_id:02d}/{SUBJECTS[-1]:02d}")
    print(f"{'#'*70}")

    # Free RAM between subjects
    clear_video_cache()
    clear_signals_cache()

    result_path = os.path.join(
        RESULTS_DIR_SHUFFLED, f'sub{subject_id:02d}_ch{CHANNEL}.pkl'
    )
    if os.path.isfile(result_path):
        print(f"  [skip] already exists: {result_path}")
        continue

    print(f"  exp_start    = {EXP_START_PER_SUBJECT[subject_id]}")
    print(f"  shuffle_seed = {SHUFFLE_SEED}")

    try:
        results = run_kfold_pipeline(
            subject_id=subject_id,
            channel_name=CHANNEL,
            time_window=TIME_WINDOW,

            # CV splits
            n_points=40, n_splits=5,

            # scaler
            scaler_method='z-score',

            # data generation
            frame_size=FRAME_SIZE,
            video_fr=FS_VIDEO,
            signal_fs=FS_EEG,
            y_type='RawEEG',

            # train
            train_overlap=TRAIN_OVERLAP,
            train_overlap_ratio=TRAIN_OVERLAP_RATIO,
            train_shuffle_output=True,
            train_signal_normalize=True,

            # test
            test_overlap=TEST_OVERLAP,
            test_overlap_ratio=TEST_OVERLAP_RATIO,
            test_shuffle_output=False,
            test_signal_normalize=True,

            # null model switch
            shuffle_pairing=True,
            shuffle_seed=SHUFFLE_SEED,

            # model
            video_length=None,
            signal_length=None,

            # training
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            loss_fn=nn.MSELoss(),
            scheduler_step_size=SCHED_STEP,
            scheduler_gamma=SCHED_GAMMA,

            # prediction
            predict_batch_size=BATCH_SIZE,

            # output
            model_exp_start=EXP_START_PER_SUBJECT[subject_id],
            save_dir=MODELS_DIR_SHUFFLED,
            device=device,
            return_history=True,
            verbose=True,
            root_dir=DATA_ROOT,
        )

    except Exception as e:
        print(f"  !! ERROR for subject={subject_id}: {e}")
        traceback.print_exc()
        continue

    results_shuffled[subject_id] = results

    with open(result_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"  [saved] {result_path}")

    if device.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
print("\n" + "="*70)
print("DONE — Null Models")
print("="*70)
print()


# ---- Trivial baselines ----
print("Trivial baselines:")
print(f"  dir: {RESULTS_DIR_TRIVIAL}")
for sid in SUBJECTS:
    p = os.path.join(RESULTS_DIR_TRIVIAL, f'sub{sid:02d}_ch{CHANNEL}.pkl')
    ok = "OK" if os.path.isfile(p) else "--"
    print(f"  [{ok}]  S{sid:02d}  {os.path.basename(p)}")


# ---- Shuffled-pairing null model ----
print()
print("Shuffled-pairing null model:")
print(f"  dir: {RESULTS_DIR_SHUFFLED}")
for sid in SUBJECTS:
    p = os.path.join(RESULTS_DIR_SHUFFLED, f'sub{sid:02d}_ch{CHANNEL}.pkl')
    ok = "OK" if os.path.isfile(p) else "--"
    print(f"  [{ok}]  S{sid:02d}  {os.path.basename(p)}")


# ---- Directories summary ----
print()
print("Directories:")
print(f"  EXP_DIR            : {EXP_DIR}")
print(f"  RESULTS_DIR_SHUFFLED: {RESULTS_DIR_SHUFFLED}")
print(f"  MODELS_DIR_SHUFFLED : {MODELS_DIR_SHUFFLED}")
print(f"  RESULTS_DIR_TRIVIAL : {RESULTS_DIR_TRIVIAL}")